In [ ]:
import random
!pip install -U scikit-learn pandas numpy matplotlib seaborn

In [ ]:
import pandas as pd
import numpy as np

In [ ]:
def load_data(path: str):
    df = pd.read_csv(path)
    return df

data = load_data("./data/Patients Data.csv")
data.columns.tolist()

In [ ]:
data.head()

In [ ]:
data["AgeCategory"].value_counts()

In [ ]:
data["State"].value_counts()

In [ ]:
def select_features(df: pd.DataFrame, features: list):
    return df[features]

filtered_data_columns = select_features(data, ["PatientID","Sex", "WeightInKilograms", "HeightInMeters", "AgeCategory"])

In [ ]:
def preprocess_data(df: pd.DataFrame):
    df.loc[:,"AgeCategory"]=df["AgeCategory"].apply(lambda x: int(random.randrange(
        int(x.strip().split(" ")[1]),
         100 if (x.strip().split(" ")[3] == 'older') else int(x.strip().split(" ")[3])
    )))

    return df
preprocessed_data = preprocess_data(filtered_data_columns)
preprocessed_data



In [ ]:
def schnider_model(age: int, weight: float, height: float, sex: str) -> dict:
    sex = sex.lower()
    if sex not in ("male", "female"):
        raise ValueError("sex must be 'male' or 'female'")

    # --- Lean Body Mass (LBM) ---
    if sex == "male":
        lbm = 1.10 * weight - 128 * (weight ** 2) / (height ** 2)
    else:
        lbm = 1.07 * weight - 148 * (weight ** 2) / (height ** 2)

    # --- Compartment volumes (L) ---
    V1 = 4.27
    V2 = 18.9 - 0.391 * (age - 53)
    V3 = 238.0

    # --- Rate constants (min-1) ---
    k10 = 0.443 + 0.0107 * (weight - 77) - 0.0159 * (lbm - 59) + 0.0062 * (height - 177)
    k12 = 0.302 - 0.0056 * (age - 53)
    k13 = 0.196
    k21 = (1.29 - 0.024 * (age - 53)) / V2
    k31 = 0.0035
    ke0 = 0.456

    # --- State-space matrices ---
    # State: [C1, C2, C3, Ce]
    A = np.array([
        [-(k10 + k12 + k13),  k21,  k31,   0   ],
        [        k12,         -k21,   0,    0   ],
        [        k13,          0,   -k31,   0   ],
        [        ke0,          0,    0,   -ke0  ]
    ])

    # Input: u(t) in µg/min → enters central compartment (C1 = x1/V1)
    B = np.array([[1 / V1], [0], [0], [0]])

    # Output: effect concentration Ce (4th state)
    C = np.array([[0, 0, 0, 1]])

    return {
        "LBM":  round(lbm, 4),
        "V1":   V1,
        "V2":   round(V2, 4),
        "V3":   V3,
        "k10":  round(k10, 6),
        "k12":  round(k12, 6),
        "k13":  k13,
        "k21":  round(k21, 6),
        "k31":  k31,
        "ke0":  ke0,
        "A":    A,
        "B":    B,
        "C":    C,
    }


def apply_schnider_model_to_dataframe(df: pd.DataFrame) -> pd.DataFrame:
    """
    Apply the Schnider model to each row of the dataframe.
    Expects columns: AgeCategory, WeightInKilograms, HeightInMeters, Sex
    """
    results = []

    for idx, row in df.iterrows():
        try:
            age = int(row["AgeCategory"])
            weight = float(row["WeightInKilograms"])
            height = float(row["HeightInMeters"])
            sex = row["Sex"]

            model_params = schnider_model(age, weight, height, sex)
            results.append(model_params)
        except Exception as e:
            print(f"Error processing row {idx}: {e}")
            results.append(None)

    # Create a new dataframe with the results
    results_df = pd.DataFrame(results)

    # Combine with original dataframe
    combined_df = pd.concat([df.reset_index(drop=True), results_df], axis=1)

    return combined_df


# Apply the model to each row
schnider_results = apply_schnider_model_to_dataframe(preprocessed_data)
schnider_results["C"].value_counts()


